In [ ]:
import hashlib
import os
from pathlib import Path
import pandas as pd
from typing import List
from lib.models.reranker import Reranker

from dotenv import load_dotenv
from langchain.retrievers import  EnsembleRetriever

from langchain.tools.retriever import create_retriever_tool
from langchain_chroma import Chroma

from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser

from lib.clients.llm import SimpleLLMFactory

# для RAGAS
from ragas import EvaluationDataset, evaluate
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from ragas.llms import LangchainLLMWrapper


from lib.models.embedder import Embedder

load_dotenv()

embedder = Embedder()  # можно передать model_name="..." при желании
embeddings = embedder.embeddings  # это HuggingFaceEmbeddings из LangChain

factory = SimpleLLMFactory(temperature=0)
gpt_oss_120b = factory.create("openai/gpt-oss-120b")


# Параметры данных/индекса
# md_folder = "/Users/sergey/Desktop/Deteiling_agent/Data/cleaned"
persist_dir = "//Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed/chromadb"
collection_name = "VectorDB_deepvk_USER-bge-m3"

# Открываем (или создаём пустую) коллекцию Chroma
vectordb = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,      # важно: тот же эмбеддер, что использовался при создании
    persist_directory=persist_dir,
)

def _docs_from_chroma(db: Chroma) -> List[Document]:
    """Забираем все документы (чанки) из существующей коллекции Chroma."""
    raw = db._collection.get(include=["documents", "metadatas"])  # приватное API
    docs: List[Document] = []
    for txt, md in zip(raw.get("documents", []), raw.get("metadatas", [])):
        docs.append(Document(page_content=txt or "", metadata=md or {}))
    return docs


# --- 3) Ретривер MMR (Chroma) ---
mmr = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 40, "lambda_mult": 0.5},
)

# --- 4) Ретривер BM25 (из чанков, восстановленных из Chroma) ---
chroma_docs = _docs_from_chroma(vectordb)
bm25 = BM25Retriever.from_documents(chroma_docs)

# --- 5) Ensemble ---
ensemble = EnsembleRetriever(
    retrievers=[mmr, bm25],
    weights=[0.6, 0.4],
)


retriever_tool = create_retriever_tool(
    ensemble,
    name="retrieve_in_vectordb",
    description="Search and return information about car care, detailing and everything related to self-washing, cleaning.",
    response_format="content_and_artifact",
)


In [5]:
# --- 2. Определяем цепочку (LLM + промпт) для генерации ответа

qa_template = PromptTemplate.from_template(
    "Используя только следующий контекст, ответьте на вопрос кратко и по существу.\n\n"
    "Контекст:\n{context}\n\n"
    "Вопрос: {question}\n"
    "Ответ:"
)

qa_chain = qa_template | gpt_oss_120b | StrOutputParser()

def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(d.page_content for d in docs)

### для каждого вопроса вытаскиваем контекст ретривером, прогоняет его через цепочку генерации ответа и сохраняет результаты в список records для дальнейшей метрики/анализа.

In [ ]:
# --- 3. Загружаем ваш датасет вопросов

df = pd.read_csv("/Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed/eval_dataset.csv")  # замените на путь к вашему CSV
# ожидается, что в df есть такие столбцы:
#   "question" (вопрос)
#   "answer" (эталонный / правильный ответ)

records = []
for _, row in df.iterrows():
    question = row["question"]
    # получение документов
    docs = ensemble.get_relevant_documents(question)
    context = format_docs(docs)
    response = qa_chain.invoke({"context": context, "question": question})
    records.append({
        "user_input": question,
        "retrieved_contexts": [d.page_content for d in docs],
        "response": response,
        "reference": row["answer"]
    })

In [9]:
records

[{'user_input': 'Какой абразивности должны быть шлифовальные листы для грубой шлифовки оптики?',
  'retrieved_contexts': ['. Повторите процедуру, если необходимо. Если оптика относительно новая, можно предварительно ее не матировать. А финишный состав иногда заменяют обычной зубной пастой – но такого же блестящего эффекта от нее не дождаться. Ещё можно воспользоваться шкуркой для более грубой шлифовки. Это нужно, если поверхность ощутимо пожелтела. Запаситесь: - чистой водой; - шлифовальными листами. Абразивность последних должна быть 800-1000 и 1500-2000 – нужно две шкурки. Но перед шлифовкой проведите полировку плафона при помощи шкурки с абразивностью 2500. Также вам пригодится весь набор из предыдущего варианта – ткань, ветошь и полироль. Приступив к шлифовке, сначала используйте шкурку с меньшей абразивностью – такой лист грубее. Работайте строго в одном направлении, горизонтально или вертикально, не возите лист во всех направлениях. Постоянно мочите и шкурку, и зону обработки. То

In [11]:
def _to_str(x):
    if isinstance(x, dict):
        # частый случай: {"answer": "..."} или {"result": "..."}
        for k in ("answer", "result", "output", "text", "content"):
            if k in x:
                return str(x[k])
        return str(x)
    return str(x)

rows = []
for r in records:
    rows.append({
        "user_input": r["user_input"],
        "reference": r["reference"],
        "response": _to_str(r["response"]),
        # контексты в CSV лучше хранить одной строкой (JSON/разделитель)
        "retrieved_contexts": "\n\n---\n\n".join(r["retrieved_contexts"]),
        "num_contexts": len(r["retrieved_contexts"]),
    })

df_out = pd.DataFrame(rows)

# 3) Путь сохранения
collection_name = "VectorDB_deepvk_USER-bge-m3"
out_dir = "/Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed"          # если у вас папка называется Data
# out_dir = "data/processed"        # если вы уже перешли на нижний регистр

os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, f"{collection_name}.csv")

# 4) Сохранение
df_out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {out_path}")

✅ Saved: /Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed/VectorDB_deepvk_USER-bge-m3.csv


In [ ]:
# --- 4. Формируем EvaluationDataset и запускаем оценку
eval_ds = EvaluationDataset.from_list(records)

# оборачиваем вашу LLM для RAGAS
evaluator_llm = LangchainLLMWrapper(gpt_oss_120b)

result = evaluate(
    dataset=eval_ds,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
)

print(result)

/var/folders/h2/r9wh0x750xq4v5z33ylwrzw40000gn/T/ipykernel_21309/96319458.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(gpt_oss_120b)
Evaluating: 100%|██████████| 207/207 [12:32<00:00,  3.63s/it]


{'context_recall': 0.9412, 'faithfulness': 0.9132, 'factual_correctness(mode=f1)': 0.1684}


В документации Ragas рекомендуется использовать три метрики для оценки RAG‑систем:

Context Recall

Оценивает, какая доля релевантных документов была найдена. Метрика разбивает эталонный ответ на утверждения и проверяет, можно ли каждое утверждение вывести из извлечённого контекста . Высокий показатель говорит о том, что система не пропускает важную информацию.

Faithfulness

Проверяет, насколько факты в ответе поддерживаются извлечённым контекстом. Ответ считается «достоверным», если все его утверждения можно обосновать контекстом . Отсутствие поддержки означает галлюцинации модели.

Factual Correctness

Сравнивает фактическую точность ответа с эталонным ответом. Ragas разбивает ответ и эталон на утверждения и использует вывод на естественном языке, чтобы измерить точность, полноту и F1‑оценку . Высокие значения свидетельствуют о соответствии фактическим данным.


## Проверяем с другой бд

In [13]:
persist_dir = "//Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed/chromadb"
collection_name = "chunk300_deepvk_USER-bge-m3"

# Открываем (или создаём пустую) коллекцию Chroma
vectordb = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,      # важно: тот же эмбеддер, что использовался при создании
    persist_directory=persist_dir,
)

def _docs_from_chroma(db: Chroma) -> List[Document]:
    """Забираем все документы (чанки) из существующей коллекции Chroma."""
    raw = db._collection.get(include=["documents", "metadatas"])  # приватное API
    docs: List[Document] = []
    for txt, md in zip(raw.get("documents", []), raw.get("metadatas", [])):
        docs.append(Document(page_content=txt or "", metadata=md or {}))
    return docs


# --- 3) Ретривер MMR (Chroma) ---
mmr = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 30, "fetch_k": 40, "lambda_mult": 0.5},
)

# --- 4) Ретривер BM25 (из чанков, восстановленных из Chroma) ---
chroma_docs = _docs_from_chroma(vectordb)
bm25 = BM25Retriever.from_documents(chroma_docs)

# --- 5) Ensemble ---
ensemble = EnsembleRetriever(
    retrievers=[mmr, bm25],
    weights=[0.6, 0.4],
)

retriever_tool = create_retriever_tool(
    ensemble,
    name="retrieve_in_vectordb",
    description="Search and return information about car care, detailing and everything related to self-washing, cleaning.",
    response_format="content_and_artifact",
)

In [14]:
# --- 2. Определяем цепочку (LLM + промпт) для генерации ответа

qa_template = PromptTemplate.from_template(
    "Используя только следующий контекст, ответьте на вопрос кратко и по существу.\n\n"
    "Контекст:\n{context}\n\n"
    "Вопрос: {question}\n"
    "Ответ:"
)

qa_chain = qa_template | gpt_oss_120b | StrOutputParser()

def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(d.page_content for d in docs)

In [15]:
df = pd.read_csv("/Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed/eval_dataset.csv")  # замените на путь к вашему CSV
# ожидается, что в df есть такие столбцы:
#   "question" (вопрос)
#   "answer" (эталонный / правильный ответ)

records = []
for _, row in df.iterrows():
    question = row["question"]
    # получение документов
    docs = ensemble.get_relevant_documents(question)
    context = format_docs(docs)
    response = qa_chain.invoke({"context": context, "question": question})
    records.append({
        "user_input": question,
        "retrieved_contexts": [d.page_content for d in docs],
        "response": response,
        "reference": row["answer"]
    })

In [16]:
def _to_str(x):
    if isinstance(x, dict):
        # частый случай: {"answer": "..."} или {"result": "..."}
        for k in ("answer", "result", "output", "text", "content"):
            if k in x:
                return str(x[k])
        return str(x)
    return str(x)

rows = []
for r in records:
    rows.append({
        "user_input": r["user_input"],
        "reference": r["reference"],
        "response": _to_str(r["response"]),
        # контексты в CSV лучше хранить одной строкой (JSON/разделитель)
        "retrieved_contexts": "\n\n---\n\n".join(r["retrieved_contexts"]),
        "num_contexts": len(r["retrieved_contexts"]),
    })

df_out = pd.DataFrame(rows)

# 3) Путь сохранения
collection_name = "chunk300_deepvk_USER-bge-m3_without_reranker"
out_dir = "/Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed"          # если у вас папка называется Data
# out_dir = "data/processed"        # если вы уже перешли на нижний регистр

os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, f"{collection_name}.csv")

# 4) Сохранение
df_out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {out_path}")

✅ Saved: /Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed/chunk300_deepvk_USER-bge-m3_without_reranker.csv


In [18]:
# --- 4. Формируем EvaluationDataset и запускаем оценку
eval_ds = EvaluationDataset.from_list(records)

# оборачиваем вашу LLM для RAGAS
evaluator_llm = LangchainLLMWrapper(gpt_oss_120b)

result = evaluate(
    dataset=eval_ds,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
)

print(result)

/var/folders/h2/r9wh0x750xq4v5z33ylwrzw40000gn/T/ipykernel_3809/1407634474.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(gpt_oss_120b)
Evaluating:  30%|███       | 63/207 [04:02<11:10,  4.66s/it]Exception raised in Job[50]: OutputParserException(Failed to parse NLIStatementOutput from completion {}. Got: 1 validation error for NLIStatementOutput
statements
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE )
Evaluating: 100%|██████████| 207/207 [12:35<00:00,  3.65s/it]


{'context_recall': 0.9710, 'faithfulness': 0.8918, 'factual_correctness(mode=f1)': 0.2031}


### БД с реранкером

In [26]:
persist_dir = "/Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed/chromadb"
collection_name = "chunk300_deepvk_USER-bge-m3"

vectordb = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=persist_dir,
)

def _docs_from_chroma(db: Chroma) -> List[Document]:
    raw = db._collection.get(include=["documents", "metadatas"])  # приватное API
    docs: List[Document] = []
    for txt, md in zip(raw.get("documents", []), raw.get("metadatas", [])):
        docs.append(Document(page_content=txt or "", metadata=md or {}))
    return docs

mmr = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 30, "fetch_k": 40, "lambda_mult": 0.5},
)

bm25 = BM25Retriever.from_documents(_docs_from_chroma(vectordb))

ensemble = EnsembleRetriever(
    retrievers=[mmr, bm25],
    weights=[0.6, 0.4],
)

reranker = Reranker("naver/xprovence-reranker-bgem3-v1")


# ---- МИНИМАЛЬНАЯ ОБЁРТКА ДЛЯ create_retriever_tool ----
class RerankWrapper:
    def __init__(self, base, rr: Reranker, fetch_k: int = 30, top_k: int = 8):
        self.base = base
        self.rr = rr
        self.fetch_k = fetch_k
        self.top_k = top_k

    def get_relevant_documents(self, query: str) -> List[Document]:
        docs = self.base.get_relevant_documents(query)[: self.fetch_k]
        return self.rr.rerank(query, docs, top_k=self.top_k)


rerank_retriever = RerankWrapper(
    base=ensemble,
    rr=reranker,
    fetch_k=30,
    top_k=5,
)

retriever_tool = create_retriever_tool(
    rerank_retriever,
    name="retrieve_in_vectordb",
    description="Search & rerank information about car care, detailing, self-washing, cleaning.",
    response_format="content_and_artifact",
)

In [27]:
# --- 2. Определяем цепочку (LLM + промпт) для генерации ответа

qa_template = PromptTemplate.from_template(
    "Используя только следующий контекст, ответьте на вопрос кратко и по существу.\n\n"
    "Контекст:\n{context}\n\n"
    "Вопрос: {question}\n"
    "Ответ:"
)

qa_chain = qa_template | gpt_oss_120b | StrOutputParser()

def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(d.page_content for d in docs)

In [28]:
df = pd.read_csv("/Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed/eval_dataset.csv")  # замените на путь к вашему CSV
# ожидается, что в df есть такие столбцы:
#   "question" (вопрос)
#   "answer" (эталонный / правильный ответ)

records = []
for _, row in df.iterrows():
    question = row["question"]
    # получение документов
    docs = ensemble.get_relevant_documents(question)
    context = format_docs(docs)
    response = qa_chain.invoke({"context": context, "question": question})
    records.append({
        "user_input": question,
        "retrieved_contexts": [d.page_content for d in docs],
        "response": response,
        "reference": row["answer"]
    })

In [29]:
def _to_str(x):
    if isinstance(x, dict):
        # частый случай: {"answer": "..."} или {"result": "..."}
        for k in ("answer", "result", "output", "text", "content"):
            if k in x:
                return str(x[k])
        return str(x)
    return str(x)

rows = []
for r in records:
    rows.append({
        "user_input": r["user_input"],
        "reference": r["reference"],
        "response": _to_str(r["response"]),
        # контексты в CSV лучше хранить одной строкой (JSON/разделитель)
        "retrieved_contexts": "\n\n---\n\n".join(r["retrieved_contexts"]),
        "num_contexts": len(r["retrieved_contexts"]),
    })

df_out = pd.DataFrame(rows)

# 3) Путь сохранения
collection_name = "chunk300_deepvk_USER-bge-m3_with_reranker"
out_dir = "/Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed"          # если у вас папка называется Data
# out_dir = "data/processed"        # если вы уже перешли на нижний регистр

os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, f"{collection_name}.csv")

# 4) Сохранение
df_out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {out_path}")

✅ Saved: /Users/sergey/Desktop/LangGraph-DIY-car-agent/Data/processed/chunk300_deepvk_USER-bge-m3_with_reranker.csv


In [30]:
# --- 4. Формируем EvaluationDataset и запускаем оценку
eval_ds = EvaluationDataset.from_list(records)

# оборачиваем вашу LLM для RAGAS
evaluator_llm = LangchainLLMWrapper(gpt_oss_120b)

result = evaluate(
    dataset=eval_ds,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
)

print(result)

/var/folders/h2/r9wh0x750xq4v5z33ylwrzw40000gn/T/ipykernel_3809/1407634474.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(gpt_oss_120b)
Evaluating: 100%|██████████| 207/207 [09:16<00:00,  2.69s/it]


{'context_recall': 0.9855, 'faithfulness': 0.9169, 'factual_correctness(mode=f1)': 0.2296}
